In [ ]:
# ENVIRONMENT SETUP & GPU CONFIGURATION

import os
import sys
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
from pathlib import Path
import json
from datetime import datetime
import gc

# TensorFlow setup
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
 try:
 for gpu in gpus:
 tf.config.experimental.set_memory_growth(gpu, True)
 print(f" Found {len(gpus)} GPU(s)")
 
 # Get GPU details
 gpu = gpus[0]
 gpu_details = tf.config.experimental.get_device_details(gpu)
 GPU_NAME = gpu_details.get('device_name', 'Unknown GPU')
 
 # Detect GPU memory
 try:
 result =!nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits 2>/dev/null
 GPU_MEMORY_GB = int(result[0].strip()) // 1024 if result else 12
 except:
 GPU_MEMORY_GB = 12
 
 print(f" GPU: {GPU_NAME}")
 print(f" Memory: {GPU_MEMORY_GB} GB")
 
 # Detect GPU type
 if 'TITAN X' in GPU_NAME.upper() or 'TITAN-X' in GPU_NAME.upper():
 GPU_TYPE = 'TITAN_X'
 elif 'T4' in GPU_NAME.upper():
 GPU_TYPE = 'T4'
 elif 'A100' in GPU_NAME.upper():
 GPU_TYPE = 'A100'
 else:
 GPU_TYPE = 'GENERIC'
 print(f" Type: {GPU_TYPE}")
 
 except RuntimeError as e:
 print(f" GPU setup error: {e}")
 GPU_MEMORY_GB = 12
 GPU_TYPE = 'TITAN_X'
 GPU_NAME = 'Unknown'
else:
 print(" No GPU found, using CPU")
 GPU_MEMORY_GB = 0
 GPU_TYPE = 'CPU'
 GPU_NAME = 'CPU'

# Detect environment
try:
 import google.colab
 IN_COLAB = True
 print(" Running in Google Colab")
except:
 IN_COLAB = False
 print(" Running locally")

# Keras imports
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import (
 ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, 
 LearningRateScheduler, TensorBoard
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1_l2

print("\n All imports successful")

In [ ]:
# V10 PAPER-SPEC CONFIGURATION
# Key changes from V9:
# - Dense units: 2048 (paper spec)
# - Gradient accumulation for effective batch=32
# - L1/L2: 0.01 (paper spec)
# - Dropout: 0.5 (paper spec)
# - Reuses V9 cache (same CLAHE preprocessing)

class Config:
 """
 V10 PAPER-SPEC Configuration
 
 Implements full paper specifications using gradient accumulation
 to overcome TITAN X memory limitations.
 """
 
 def __init__(self, gpu_memory_gb=12):
 self.GPU_MEMORY_GB = gpu_memory_gb
 self.VERSION = "V10_PAPER_SPEC"
 
 # Detect environment
 try:
 import google.colab
 self.IN_COLAB = True
 except:
 self.IN_COLAB = False
 
 # PATHS - Use V9 cache for speed!
 if self.IN_COLAB:
 self.BASE_PATH = Path('/content/drive/MyDrive/CBIS-DDSM-data')
 self.DICOM_PATH = self.BASE_PATH / 'CBIS-DDSM'
 self.CSV_PATH = self.BASE_PATH / 'csv'
 self.OUTPUT_PATH = self.BASE_PATH / 'model_outputs_v10'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'checkpoints_v10'
 self.RESULTS_PATH = self.BASE_PATH / 'results_v10'
 # Reuse V9 cache (same preprocessing)
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache_v9'
 else:
 self.BASE_PATH = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
 self.DICOM_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
 self.CSV_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
 self.OUTPUT_PATH = self.BASE_PATH / 'local_outputs_v10'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'local_checkpoints_v10'
 self.RESULTS_PATH = self.BASE_PATH / 'local_results_v10'
 # REUSE V9 CACHE - Same CLAHE preprocessing!
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache_v9'
 
 # IMAGE CONFIGURATION (same as V9 for cache compatibility)
 self.IMG_SIZE = (224, 224) # Paper spec
 self.IMG_CHANNELS = 3
 
 # CLAHE settings (same as V9 cache)
 self.USE_CLAHE = True
 self.CLAHE_CLIP_LIMIT = 2.0
 self.CLAHE_TILE_SIZE = (8, 8)
 
 # V10 PAPER-SPEC SETTINGS WITH GRADIENT ACCUMULATION
 
 # PAPER SPEC: Dense units = 2048
 self.DENSE_UNITS = 2048 # Up from 1024!
 
 # Gradient accumulation to achieve effective batch=32
 if gpu_memory_gb >= 16:
 # High memory GPU: can use larger physical batch
 self.BATCH_SIZE = 8
 self.ACCUMULATION_STEPS = 4 # 8 × 4 = 32
 self.GPU_PROFILE = "HIGH_MEMORY"
 elif gpu_memory_gb >= 12:
 # TITAN X: use smaller batch with more accumulation
 self.BATCH_SIZE = 4 # Fits with 2048 dense
 self.ACCUMULATION_STEPS = 8 # 4 × 8 = 32 (paper spec!)
 self.GPU_PROFILE = "TITAN_X"
 else:
 # Low memory: even smaller batch
 self.BATCH_SIZE = 2
 self.ACCUMULATION_STEPS = 16 # 2 × 16 = 32
 self.GPU_PROFILE = "LOW_MEMORY"
 
 self.EFFECTIVE_BATCH_SIZE = self.BATCH_SIZE * self.ACCUMULATION_STEPS
 
 # Single model (ensemble would require 3× memory)
 self.NUM_ENSEMBLE_MODELS = 1
 self.USE_MIXED_PRECISION = False # CC 5.2 doesn't benefit
 self.ENSEMBLE_SEEDS = [42]
 
 # TEST-TIME AUGMENTATION
 self.USE_TTA = True
 self.TTA_AUGMENTATIONS = 4
 
 # TRAINING PARAMETERS
 self.STAGE1_EPOCHS = 25
 self.STAGE1_LR = 1e-3
 
 self.STAGE2_EPOCHS = 40
 self.STAGE2_LR = 5e-4
 
 self.STAGE3_EPOCHS = 80
 self.STAGE3_LR = 1e-4 # Paper spec: 0.0001
 
 # LR Warmup (proven beneficial in V9)
 self.USE_LR_WARMUP = True
 self.WARMUP_EPOCHS = 5
 self.WARMUP_START_LR = 1e-7
 
 self.TRAIN_SPLIT = 0.70
 self.VAL_SPLIT = 0.15
 self.TEST_SPLIT = 0.15
 
 # PAPER-SPEC REGULARIZATION (KEY CHANGE!)
 self.L1_REGULARIZATION = 0.01 # Paper spec! (was 0.0001)
 self.L2_REGULARIZATION = 0.01 # Paper spec! (was 0.0001)
 self.DROPOUT_RATE = 0.5 # Paper spec! (was 0.25)
 self.LABEL_SMOOTHING = 0.1
 self.GRADIENT_CLIPNORM = 1.0
 
 # CLASS WEIGHTING
 self.USE_CLASS_WEIGHT = True
 self.MALIGNANT_WEIGHT_MULTIPLIER = 2.5 # Balanced from V9
 
 # FOCAL LOSS
 self.USE_FOCAL_LOSS = True
 self.FOCAL_LOSS_ALPHA = 0.70
 self.FOCAL_LOSS_GAMMA = 2.0
 
 # ARCHITECTURE
 self.FREEZE_STAGE2 = 300
 self.FREEZE_STAGE3 = 100
 
 # CALLBACKS
 self.EARLY_STOP_PATIENCE = 20
 self.LR_REDUCE_PATIENCE = 8
 self.LR_REDUCE_FACTOR = 0.5
 self.MIN_EPOCHS_STAGE1 = 15
 self.MIN_EPOCHS_STAGE2 = 25
 self.MIN_EPOCHS_STAGE3 = 40
 
 # PERFORMANCE OPTIMIZATION
 self.PREFETCH_BUFFER = tf.data.AUTOTUNE
 self.SHUFFLE_BUFFER = 1000
 self.NUM_PARALLEL_CALLS = tf.data.AUTOTUNE
 
 self._create_dirs()
 
 def _create_dirs(self):
 for path in [self.OUTPUT_PATH, self.CHECKPOINT_PATH, self.RESULTS_PATH]:
 path.mkdir(parents=True, exist_ok=True)
 
 def print_config(self):
 print("\n" + "="*70)
 print(" V10 PAPER-SPEC CONFIGURATION")
 print("="*70)
 
 print(f"\n GPU Profile: {self.GPU_PROFILE} ({self.GPU_MEMORY_GB}GB)")
 print(f" Physical Batch: {self.BATCH_SIZE}")
 print(f" Accum Steps: {self.ACCUMULATION_STEPS}")
 print(f" Effective Batch: {self.EFFECTIVE_BATCH_SIZE} (paper spec: 32) ")
 
 print(f"\n Architecture (PAPER SPEC):")
 print(f" Dense Units: {self.DENSE_UNITS} (paper: 2048) ")
 print(f" L1/L2 Reg: {self.L1_REGULARIZATION} (paper: 0.01) ")
 print(f" Dropout: {self.DROPOUT_RATE} (paper: 0.5) ")
 
 print(f"\n Environment: {'Google Colab' if self.IN_COLAB else 'Local Machine'}")
 print(f" Cache: {self.CACHE_PATH}")
 print(f" Checkpoints: {self.CHECKPOINT_PATH}")
 
 print(f"\n Training Strategy:")
 print(f" Stage 1: {self.STAGE1_EPOCHS} epochs @ LR={self.STAGE1_LR}")
 print(f" Stage 2: {self.STAGE2_EPOCHS} epochs @ LR={self.STAGE2_LR}")
 print(f" Stage 3: {self.STAGE3_EPOCHS} epochs @ LR={self.STAGE3_LR}")
 
 # Check cache
 cache_files = ['train_cache_v9.npz', 'val_cache_v9.npz', 'test_cache_v9.npz']
 cache_exists = all((self.CACHE_PATH / f).exists() for f in cache_files)
 if cache_exists:
 print(f"\n V9 CACHE FOUND - Reusing preprocessed data!")
 total_size = sum((self.CACHE_PATH / f).stat().st_size for f in cache_files) / 1024**2
 print(f" Cache size: {total_size:.1f} MB")
 else:
 print(f"\n No V9 cache found - Will need to preprocess")
 
 print("="*70 + "\n")

# Initialize configuration
config = Config(gpu_memory_gb=GPU_MEMORY_GB)
config.print_config()

print(" Using FP32 (mixed precision disabled for TITAN X compatibility)")

In [ ]:
# LOAD PREPROCESSED CACHE FROM V9
# V10 reuses V9 cache (same CLAHE preprocessing)
# This saves ~10 minutes of preprocessing time!

print("="*70)
print(" LOADING V9 PREPROCESSED CACHE")
print("="*70)

cache_files = {
 'train': config.CACHE_PATH / 'train_cache_v9.npz',
 'val': config.CACHE_PATH / 'val_cache_v9.npz',
 'test': config.CACHE_PATH / 'test_cache_v9.npz'
}

# Check cache exists
cache_ready = all(f.exists() for f in cache_files.values())

if cache_ready:
 print("\n Loading preprocessed data from V9 cache...")
 
 # Load train
 train_cache = np.load(cache_files['train'])
 train_images = train_cache['images']
 train_labels = train_cache['labels']
 print(f" Train: {len(train_images)} samples")
 
 # Load val
 val_cache = np.load(cache_files['val'])
 val_images = val_cache['images']
 val_labels = val_cache['labels']
 print(f" Val: {len(val_images)} samples")
 
 # Load test
 test_cache = np.load(cache_files['test'])
 test_images = test_cache['images']
 test_labels = test_cache['labels']
 print(f" Test: {len(test_images)} samples")
 
 # Summary
 total_samples = len(train_images) + len(val_images) + len(test_images)
 print(f"\n Total: {total_samples} samples loaded")
 print(f" Image shape: {train_images.shape[1:]}")
 print(f" Data type: {train_images.dtype}")
 
 # Class distribution
 n_benign = int(np.sum(train_labels == 0))
 n_malignant = int(np.sum(train_labels == 1))
 print(f"\n Training class distribution:")
 print(f" Benign: {n_benign} ({100*n_benign/len(train_labels):.1f}%)")
 print(f" Malignant: {n_malignant} ({100*n_malignant/len(train_labels):.1f}%)")
 
else:
 print("\n V9 cache not found!")
 print(" Please run V9 notebook first to create the cache,")
 print(" or the next cells will preprocess from scratch.")
 train_images, train_labels = None, None
 val_images, val_labels = None, None
 test_images, test_labels = None, None

print("\n" + "="*70)

In [ ]:
# CREATE TF DATASETS WITH GRADIENT ACCUMULATION SUPPORT

print("="*70)
print(" CREATING TF DATASETS")
print("="*70)

# Data augmentation function
@tf.function
def augment_batch(images, labels):
 """Apply training augmentation"""
 images = tf.image.random_flip_left_right(images)
 images = tf.image.random_flip_up_down(images)
 images = tf.image.random_brightness(images, 0.1)
 images = tf.image.random_contrast(images, 0.9, 1.1)
 
 # Random rotation (0, 90, 180, 270 degrees)
 k = tf.random.uniform([], 0, 4, dtype=tf.int32)
 images = tf.image.rot90(images, k)
 
 return images, labels

def create_dataset(images, labels, batch_size, is_training=False):
 """Create tf.data.Dataset optimized for gradient accumulation"""
 dataset = tf.data.Dataset.from_tensor_slices((images, labels))
 
 if is_training:
 dataset = dataset.shuffle(config.SHUFFLE_BUFFER, reshuffle_each_iteration=True)
 dataset = dataset.batch(batch_size, drop_remainder=True)
 dataset = dataset.map(augment_batch, num_parallel_calls=config.NUM_PARALLEL_CALLS)
 else:
 dataset = dataset.batch(batch_size)
 
 dataset = dataset.prefetch(config.PREFETCH_BUFFER)
 return dataset

# Create datasets with physical batch size (not effective)
train_dataset = create_dataset(train_images, train_labels, config.BATCH_SIZE, is_training=True)
val_dataset = create_dataset(val_images, val_labels, config.BATCH_SIZE, is_training=False)
test_dataset = create_dataset(test_images, test_labels, config.BATCH_SIZE, is_training=False)

# Calculate steps
steps_per_epoch = len(train_images) // config.BATCH_SIZE
# Adjust for accumulation (each "step" requires ACCUMULATION_STEPS batches)
accumulated_steps_per_epoch = steps_per_epoch // config.ACCUMULATION_STEPS

print(f"\n Dataset Configuration:")
print(f" Physical batch size: {config.BATCH_SIZE}")
print(f" Accumulation steps: {config.ACCUMULATION_STEPS}")
print(f" Effective batch: {config.EFFECTIVE_BATCH_SIZE}")
print(f" Steps per epoch: {accumulated_steps_per_epoch} (accumulated)")
print(f" Total batches/epoch: {steps_per_epoch}")

print("\n Datasets created")
print("="*70)

In [ ]:
# CLASS WEIGHTS AND FOCAL LOSS

print("="*70)
print(" COMPUTING CLASS WEIGHTS")
print("="*70)

# Compute class weights
n_benign = int(np.sum(train_labels == 0))
n_malignant = int(np.sum(train_labels == 1))
total = n_benign + n_malignant

# Balanced weights with malignant emphasis
weight_benign = total / (2 * n_benign)
weight_malignant = (total / (2 * n_malignant)) * config.MALIGNANT_WEIGHT_MULTIPLIER

class_weights = {0: weight_benign, 1: weight_malignant}
print(f"\n Class weights:")
print(f" Benign (0): {weight_benign:.4f}")
print(f" Malignant (1): {weight_malignant:.4f}")
print(f" Ratio: {weight_malignant/weight_benign:.2f}x")

# Focal loss for handling class imbalance
@tf.function
def focal_loss_fn(y_true, y_pred):
 """Focal loss with label smoothing"""
 alpha = config.FOCAL_LOSS_ALPHA
 gamma = config.FOCAL_LOSS_GAMMA
 smoothing = config.LABEL_SMOOTHING
 
 # Label smoothing
 y_true_smooth = y_true * (1 - smoothing) + 0.5 * smoothing
 
 # Clip predictions
 y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
 
 # Focal loss
 bce = -y_true_smooth * tf.math.log(y_pred) - (1 - y_true_smooth) * tf.math.log(1 - y_pred)
 
 p_t = y_true_smooth * y_pred + (1 - y_true_smooth) * (1 - y_pred)
 alpha_t = y_true_smooth * alpha + (1 - y_true_smooth) * (1 - alpha)
 focal_weight = alpha_t * tf.pow(1 - p_t, gamma)
 
 return tf.reduce_mean(focal_weight * bce)

print("\n Focal loss configured")
print(f" Alpha: {config.FOCAL_LOSS_ALPHA}")
print(f" Gamma: {config.FOCAL_LOSS_GAMMA}")
print("="*70)

In [ ]:
# V10 MODEL ARCHITECTURE (PAPER SPEC: 2048 DENSE)

print("="*70)
print(" BUILDING V10 PAPER-SPEC MODEL")
print("="*70)

def create_model(seed=42):
 """
 Create V10 model with PAPER SPECIFICATIONS:
 - DenseNet121 backbone with ImageNet weights
 - BatchNorm → Dense(2048) → BatchNorm → Dropout(0.5) → Output
 - L1/L2 regularization = 0.01
 """
 tf.random.set_seed(seed)
 np.random.seed(seed)
 
 # Base model
 base_model = DenseNet121(
 weights='imagenet',
 include_top=False,
 input_shape=(*config.IMG_SIZE, config.IMG_CHANNELS),
 pooling='avg'
 )
 
 # Initially freeze all layers
 for layer in base_model.layers:
 layer.trainable = False
 
 # Build classification head (PAPER SPEC)
 x = base_model.output
 
 # BatchNorm before dense (paper spec)
 x = layers.BatchNormalization(name='bn_pre_dense')(x)
 
 # PAPER SPEC: Dense layer with 2048 units, L1/L2=0.01
 x = layers.Dense(
 config.DENSE_UNITS, # 2048!
 activation='relu',
 kernel_regularizer=l1_l2(l1=config.L1_REGULARIZATION, l2=config.L2_REGULARIZATION),
 name='dense_paper_spec'
 )(x)
 
 # BatchNorm after dense (paper spec)
 x = layers.BatchNormalization(name='bn_post_dense')(x)
 
 # PAPER SPEC: Dropout = 0.5
 x = layers.Dropout(config.DROPOUT_RATE, name='dropout_paper_spec')(x)
 
 # Output layer (binary classification)
 outputs = layers.Dense(1, activation='sigmoid', name='output')(x)
 
 model = Model(inputs=base_model.input, outputs=outputs)
 
 return model, base_model

# Create model
model, base_model = create_model(seed=42)

# Print architecture summary
print(f"\n Model Architecture:")
print(f" Base: DenseNet121 (ImageNet weights)")
print(f" Dense units: {config.DENSE_UNITS} (paper spec: 2048) ")
print(f" L1/L2 reg: {config.L1_REGULARIZATION} (paper spec: 0.01) ")
print(f" Dropout: {config.DROPOUT_RATE} (paper spec: 0.5) ")

# Count parameters
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\n Parameters:")
print(f" Total: {total_params:,}")
print(f" Trainable: {trainable_params:,}")

print("\n V10 paper-spec model created")
print("="*70)

In [ ]:
# GRADIENT ACCUMULATION TRAINING UTILITIES

print("="*70)
print(" SETTING UP GRADIENT ACCUMULATION TRAINING")
print("="*70)

class GradientAccumulationTrainer:
 """
 Custom trainer that accumulates gradients over multiple batches.
 This allows training with effective batch=32 on 12GB VRAM.
 """
 
 def __init__(self, model, optimizer, loss_fn, accumulation_steps):
 self.model = model
 self.optimizer = optimizer
 self.loss_fn = loss_fn
 self.accumulation_steps = accumulation_steps
 
 # Initialize gradient accumulators
 self.gradient_accumulator = [
 tf.Variable(tf.zeros_like(v), trainable=False)
 for v in model.trainable_variables
 ]
 
 # Metrics
 self.train_loss = tf.keras.metrics.Mean(name='train_loss')
 self.train_auc = tf.keras.metrics.AUC(name='train_auc')
 
 def reset_accumulators(self):
 """Reset gradient accumulators to zero"""
 for acc in self.gradient_accumulator:
 acc.assign(tf.zeros_like(acc))
 
 @tf.function
 def accumulate_gradients(self, x_batch, y_batch, sample_weights=None):
 """Compute and accumulate gradients for one batch"""
 with tf.GradientTape() as tape:
 predictions = self.model(x_batch, training=True)
 
 # Compute loss
 loss = self.loss_fn(y_batch, predictions)
 
 # Apply sample weights if provided
 if sample_weights is not None:
 loss = loss * sample_weights
 loss = tf.reduce_mean(loss)
 
 # Scale loss by accumulation steps
 scaled_loss = loss / self.accumulation_steps
 
 # Compute gradients
 gradients = tape.gradient(scaled_loss, self.model.trainable_variables)
 
 # Accumulate gradients
 for acc, grad in zip(self.gradient_accumulator, gradients):
 if grad is not None:
 acc.assign_add(grad)
 
 return loss, predictions
 
 @tf.function
 def apply_accumulated_gradients(self):
 """Apply accumulated gradients with clipping"""
 # Clip gradients
 clipped_gradients = [
 tf.clip_by_norm(g, config.GRADIENT_CLIPNORM)
 for g in self.gradient_accumulator
 ]
 
 # Apply gradients
 self.optimizer.apply_gradients(
 zip(clipped_gradients, self.model.trainable_variables)
 )
 
 def train_epoch(self, dataset, class_weights=None):
 """Train for one epoch with gradient accumulation"""
 self.train_loss.reset_state()
 self.train_auc.reset_state()
 
 step_count = 0
 batch_count = 0
 
 for x_batch, y_batch in dataset:
 # Compute sample weights
 if class_weights is not None:
 sample_weights = tf.where(
 y_batch == 1,
 class_weights[1],
 class_weights[0]
 )
 else:
 sample_weights = None
 
 # Accumulate gradients
 loss, preds = self.accumulate_gradients(x_batch, y_batch, sample_weights)
 
 # Update metrics
 self.train_loss.update_state(loss)
 self.train_auc.update_state(y_batch, preds)
 
 batch_count += 1
 
 # Apply gradients after accumulation_steps batches
 if batch_count >= self.accumulation_steps:
 self.apply_accumulated_gradients()
 self.reset_accumulators()
 batch_count = 0
 step_count += 1
 
 return self.train_loss.result().numpy(), self.train_auc.result().numpy()

print(f"\n Gradient accumulation trainer configured")
print(f" Physical batch: {config.BATCH_SIZE}")
print(f" Accumulation: {config.ACCUMULATION_STEPS}")
print(f" Effective batch: {config.EFFECTIVE_BATCH_SIZE}")
print("="*70)

In [ ]:
# LEARNING RATE SCHEDULE WITH WARMUP

print("="*70)
print(" CONFIGURING LEARNING RATE SCHEDULE")
print("="*70)

def create_lr_schedule(base_lr, total_epochs, warmup_epochs=5, warmup_start_lr=1e-7):
 """
 Create learning rate schedule with warmup.
 
 Warmup: Linear increase from warmup_start_lr to base_lr
 Then: Cosine decay
 """
 def lr_schedule(epoch):
 if epoch < warmup_epochs:
 # Linear warmup
 return warmup_start_lr + (base_lr - warmup_start_lr) * (epoch / warmup_epochs)
 else:
 # Cosine decay after warmup
 progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
 return base_lr * (1 + np.cos(np.pi * progress)) / 2
 
 return lr_schedule

print(f"\n LR Schedule configured")
print(f" Warmup epochs: {config.WARMUP_EPOCHS}")
print(f" Warmup start: {config.WARMUP_START_LR}")
print("="*70)

In [ ]:
# 3-STAGE TRAINING FUNCTION

print("="*70)
print(" DEFINING 3-STAGE TRAINING FUNCTION")
print("="*70)

def evaluate_model(model, dataset):
 """Evaluate model and return loss and AUC"""
 val_loss_metric = tf.keras.metrics.Mean()
 val_auc_metric = tf.keras.metrics.AUC()
 
 for x_batch, y_batch in dataset:
 preds = model(x_batch, training=False)
 loss = focal_loss_fn(y_batch, preds)
 val_loss_metric.update_state(loss)
 val_auc_metric.update_state(y_batch, preds)
 
 return val_loss_metric.result().numpy(), val_auc_metric.result().numpy()

def train_model(model, base_model, train_ds, val_ds, class_weights):
 """
 V10 3-stage training with gradient accumulation.
 
 Stage 1: Train only classification head
 Stage 2: Unfreeze top layers of backbone
 Stage 3: Fine-tune more layers with lower LR
 """
 histories = {'stage1': [], 'stage2': [], 'stage3': []}
 best_val_auc = 0.0
 best_weights = None
 
 # Convert class weights to tensors
 class_weights_tensor = {
 0: tf.constant(class_weights[0], dtype=tf.float32),
 1: tf.constant(class_weights[1], dtype=tf.float32)
 }
 
 # STAGE 1: Train classification head only
 print("\n" + "="*70)
 print(" STAGE 1: Training Classification Head")
 print("="*70)
 print(f" Epochs: {config.STAGE1_EPOCHS}")
 print(f" LR: {config.STAGE1_LR}")
 print(f" Trainable layers: Classification head only")
 
 # Freeze backbone
 for layer in base_model.layers:
 layer.trainable = False
 
 # Create optimizer and trainer
 lr_schedule = create_lr_schedule(
 config.STAGE1_LR, config.STAGE1_EPOCHS,
 config.WARMUP_EPOCHS, config.WARMUP_START_LR
 )
 optimizer = Adam(learning_rate=config.STAGE1_LR)
 trainer = GradientAccumulationTrainer(
 model, optimizer, focal_loss_fn, config.ACCUMULATION_STEPS
 )
 
 patience_counter = 0
 
 for epoch in range(config.STAGE1_EPOCHS):
 # Update learning rate
 current_lr = lr_schedule(epoch)
 optimizer.learning_rate.assign(current_lr)
 
 # Train
 train_loss, train_auc = trainer.train_epoch(train_ds, class_weights_tensor)
 
 # Validate
 val_loss, val_auc = evaluate_model(model, val_ds)
 
 histories['stage1'].append({
 'epoch': epoch + 1,
 'train_loss': float(train_loss),
 'train_auc': float(train_auc),
 'val_loss': float(val_loss),
 'val_auc': float(val_auc),
 'lr': float(current_lr)
 })
 
 print(f" Epoch {epoch+1:3d}/{config.STAGE1_EPOCHS} - "
 f"loss: {train_loss:.4f} - auc: {train_auc:.4f} - "
 f"val_loss: {val_loss:.4f} - val_auc: {val_auc:.4f} - lr: {current_lr:.2e}")
 
 # Save best model
 if val_auc > best_val_auc:
 best_val_auc = val_auc
 best_weights = model.get_weights()
 model.save(config.CHECKPOINT_PATH / 'stage1_best_model.h5')
 patience_counter = 0
 else:
 patience_counter += 1
 
 # Early stopping (after minimum epochs)
 if epoch >= config.MIN_EPOCHS_STAGE1 and patience_counter >= config.EARLY_STOP_PATIENCE:
 print(f" Early stopping at epoch {epoch+1}")
 break
 
 print(f"\n Stage 1 complete. Best val_auc: {best_val_auc:.4f}")
 
 # STAGE 2: Unfreeze top layers
 print("\n" + "="*70)
 print(" STAGE 2: Fine-tuning Top Layers")
 print("="*70)
 print(f" Epochs: {config.STAGE2_EPOCHS}")
 print(f" LR: {config.STAGE2_LR}")
 print(f" Unfreezing from layer {config.FREEZE_STAGE2}")
 
 # Unfreeze top layers
 for layer in base_model.layers[config.FREEZE_STAGE2:]:
 layer.trainable = True
 
 trainable_count = sum([1 for l in model.layers if l.trainable])
 print(f" Trainable layers: {trainable_count}")
 
 # New optimizer with lower LR
 lr_schedule = create_lr_schedule(
 config.STAGE2_LR, config.STAGE2_EPOCHS,
 config.WARMUP_EPOCHS, config.WARMUP_START_LR
 )
 optimizer = Adam(learning_rate=config.STAGE2_LR)
 trainer = GradientAccumulationTrainer(
 model, optimizer, focal_loss_fn, config.ACCUMULATION_STEPS
 )
 
 patience_counter = 0
 stage2_best_auc = best_val_auc
 
 for epoch in range(config.STAGE2_EPOCHS):
 current_lr = lr_schedule(epoch)
 optimizer.learning_rate.assign(current_lr)
 
 train_loss, train_auc = trainer.train_epoch(train_ds, class_weights_tensor)
 val_loss, val_auc = evaluate_model(model, val_ds)
 
 histories['stage2'].append({
 'epoch': epoch + 1,
 'train_loss': float(train_loss),
 'train_auc': float(train_auc),
 'val_loss': float(val_loss),
 'val_auc': float(val_auc),
 'lr': float(current_lr)
 })
 
 print(f" Epoch {epoch+1:3d}/{config.STAGE2_EPOCHS} - "
 f"loss: {train_loss:.4f} - auc: {train_auc:.4f} - "
 f"val_loss: {val_loss:.4f} - val_auc: {val_auc:.4f} - lr: {current_lr:.2e}")
 
 if val_auc > best_val_auc:
 best_val_auc = val_auc
 best_weights = model.get_weights()
 model.save(config.CHECKPOINT_PATH / 'stage2_best_model.h5')
 patience_counter = 0
 else:
 patience_counter += 1
 
 if epoch >= config.MIN_EPOCHS_STAGE2 and patience_counter >= config.EARLY_STOP_PATIENCE:
 print(f" Early stopping at epoch {epoch+1}")
 break
 
 print(f"\n Stage 2 complete. Best val_auc: {best_val_auc:.4f}")
 
 # STAGE 3: Fine-tune more layers
 print("\n" + "="*70)
 print(" STAGE 3: Deep Fine-tuning")
 print("="*70)
 print(f" Epochs: {config.STAGE3_EPOCHS}")
 print(f" LR: {config.STAGE3_LR} (paper spec: 0.0001)")
 print(f" Unfreezing from layer {config.FREEZE_STAGE3}")
 
 # Unfreeze more layers
 for layer in base_model.layers[config.FREEZE_STAGE3:]:
 layer.trainable = True
 
 trainable_count = sum([1 for l in model.layers if l.trainable])
 print(f" Trainable layers: {trainable_count}")
 
 # Restore best weights before Stage 3
 if best_weights is not None:
 model.set_weights(best_weights)
 print(" Restored best weights from Stage 2")
 
 lr_schedule = create_lr_schedule(
 config.STAGE3_LR, config.STAGE3_EPOCHS,
 config.WARMUP_EPOCHS, config.WARMUP_START_LR
 )
 optimizer = Adam(learning_rate=config.STAGE3_LR)
 trainer = GradientAccumulationTrainer(
 model, optimizer, focal_loss_fn, config.ACCUMULATION_STEPS
 )
 
 patience_counter = 0
 
 for epoch in range(config.STAGE3_EPOCHS):
 current_lr = lr_schedule(epoch)
 optimizer.learning_rate.assign(current_lr)
 
 train_loss, train_auc = trainer.train_epoch(train_ds, class_weights_tensor)
 val_loss, val_auc = evaluate_model(model, val_ds)
 
 histories['stage3'].append({
 'epoch': epoch + 1,
 'train_loss': float(train_loss),
 'train_auc': float(train_auc),
 'val_loss': float(val_loss),
 'val_auc': float(val_auc),
 'lr': float(current_lr)
 })
 
 print(f" Epoch {epoch+1:3d}/{config.STAGE3_EPOCHS} - "
 f"loss: {train_loss:.4f} - auc: {train_auc:.4f} - "
 f"val_loss: {val_loss:.4f} - val_auc: {val_auc:.4f} - lr: {current_lr:.2e}")
 
 if val_auc > best_val_auc:
 best_val_auc = val_auc
 best_weights = model.get_weights()
 model.save(config.CHECKPOINT_PATH / 'stage3_best_model.h5')
 patience_counter = 0
 else:
 patience_counter += 1
 
 if epoch >= config.MIN_EPOCHS_STAGE3 and patience_counter >= config.EARLY_STOP_PATIENCE:
 print(f" Early stopping at epoch {epoch+1}")
 break
 
 # Restore best weights
 if best_weights is not None:
 model.set_weights(best_weights)
 
 print(f"\n Stage 3 complete. Best val_auc: {best_val_auc:.4f}")
 
 # Save final model
 model.save(config.OUTPUT_PATH / 'v10_final_model.h5')
 print(f"\n Final model saved to {config.OUTPUT_PATH / 'v10_final_model.h5'}")
 
 return model, histories, best_val_auc

print("\n Training function defined")
print("="*70)

In [ ]:
# RUN TRAINING

print("\n" + "="*70)
print(" STARTING V10 PAPER-SPEC TRAINING")
print("="*70)
print(f"\n Configuration:")
print(f" Dense units: {config.DENSE_UNITS} (paper spec)")
print(f" Effective batch: {config.EFFECTIVE_BATCH_SIZE} (paper spec)")
print(f" L1/L2 reg: {config.L1_REGULARIZATION} (paper spec)")
print(f" Dropout: {config.DROPOUT_RATE} (paper spec)")
print(f"\n⏱ Estimated time: ~90-120 minutes on TITAN X")
print("="*70 + "\n")

# Clear GPU memory
gc.collect()
tf.keras.backend.clear_session()

# Recreate model (fresh start)
model, base_model = create_model(seed=42)

# Start training
start_time = datetime.now()

trained_model, histories, best_auc = train_model(
 model, base_model, train_dataset, val_dataset, class_weights
)

end_time = datetime.now()
training_duration = end_time - start_time

print("\n" + "="*70)
print(" TRAINING COMPLETE")
print("="*70)
print(f" Duration: {training_duration}")
print(f" Best validation AUC: {best_auc:.4f}")
print("="*70)

In [ ]:
# TEST-TIME AUGMENTATION EVALUATION

print("="*70)
print(" EVALUATING WITH TEST-TIME AUGMENTATION")
print("="*70)

def apply_tta(model, images, num_augmentations=4):
 """Apply test-time augmentation and average predictions"""
 all_preds = []
 
 # Original
 preds = model.predict(images, verbose=0)
 all_preds.append(preds)
 
 # Horizontal flip
 flipped = np.flip(images, axis=2)
 preds = model.predict(flipped, verbose=0)
 all_preds.append(preds)
 
 if num_augmentations >= 4:
 # Vertical flip
 flipped = np.flip(images, axis=1)
 preds = model.predict(flipped, verbose=0)
 all_preds.append(preds)
 
 # Both flips
 flipped = np.flip(np.flip(images, axis=1), axis=2)
 preds = model.predict(flipped, verbose=0)
 all_preds.append(preds)
 
 # Average predictions
 return np.mean(all_preds, axis=0)

# Evaluate on test set with TTA
print("\n Applying TTA with 4 augmentations...")
y_pred_tta = apply_tta(trained_model, test_images, config.TTA_AUGMENTATIONS)
y_true = test_labels

# Compute AUC
from sklearn.metrics import roc_auc_score, roc_curve
auc = roc_auc_score(y_true, y_pred_tta)
print(f"\n Test AUC-ROC (with TTA): {auc:.4f}")

# Find optimal threshold using Youden's J
fpr, tpr, thresholds = roc_curve(y_true, y_pred_tta)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f" Optimal threshold (Youden's J): {optimal_threshold:.4f}")

# Compute metrics at optimal threshold
y_pred_binary = (y_pred_tta >= optimal_threshold).astype(int).flatten()

tp = np.sum((y_pred_binary == 1) & (y_true == 1))
tn = np.sum((y_pred_binary == 0) & (y_true == 0))
fp = np.sum((y_pred_binary == 1) & (y_true == 0))
fn = np.sum((y_pred_binary == 0) & (y_true == 1))

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"\n Metrics at optimal threshold ({optimal_threshold:.3f}):")
print(f" Sensitivity: {100*sensitivity:.1f}%")
print(f" Specificity: {100*specificity:.1f}%")
print(f" Accuracy: {100*accuracy:.1f}%")
print(f" PPV: {100*ppv:.1f}%")
print(f" NPV: {100*npv:.1f}%")

print("\n" + "="*70)

In [ ]:
# COMPARISON WITH PREVIOUS VERSIONS

print("="*70)
print(" PERFORMANCE COMPARISON")
print("="*70)

# Historical results
versions = {
 'V4': {'auc': 0.707, 'sensitivity': 0.583, 'specificity': 0.762},
 'V6': {'auc': 0.723, 'sensitivity': 0.633, 'specificity': 0.698},
 'V7': {'auc': 0.732, 'sensitivity': 0.697, 'specificity': 0.660},
 'V9': {'auc': 0.738, 'sensitivity': 0.587, 'specificity': 0.762},
 'V10': {'auc': auc, 'sensitivity': sensitivity, 'specificity': specificity}
}

print("\n{:<8} {:<10} {:<12} {:<12}".format('Version', 'AUC', 'Sensitivity', 'Specificity'))
print("-" * 45)
for v, m in versions.items():
 print("{:<8} {:<10.4f} {:<12.1%} {:<12.1%}".format(v, m['auc'], m['sensitivity'], m['specificity']))

# Improvement over V9
v9_auc = versions['V9']['auc']
improvement = (auc - v9_auc) / v9_auc * 100

print(f"\n V10 vs V9:")
print(f" AUC change: {auc - v9_auc:+.4f} ({improvement:+.2f}%)")
print(f" Sensitivity change: {sensitivity - versions['V9']['sensitivity']:+.1%}")
print(f" Specificity change: {specificity - versions['V9']['specificity']:+.1%}")

print("\n" + "="*70)

In [ ]:
# VISUALIZATIONS

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

print("="*70)
print(" GENERATING VISUALIZATIONS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. ROC Curve
ax1 = axes[0, 0]
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'V10 (AUC = {auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax1.scatter([fpr[optimal_idx]], [tpr[optimal_idx]], color='red', s=100, 
 marker='*', zorder=5, label='Operating Point')
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('V10 ROC Curve (Paper Spec Configuration)')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# 2. Prediction Distribution
ax2 = axes[0, 1]
ax2.hist(y_pred_tta[y_true == 0], bins=50, alpha=0.6, label='Benign', color='blue', density=True)
ax2.hist(y_pred_tta[y_true == 1], bins=50, alpha=0.6, label='Malignant', color='red', density=True)
ax2.axvline(optimal_threshold, color='green', linestyle='--', linewidth=2, 
 label=f'Threshold = {optimal_threshold:.3f}')
ax2.set_xlabel('Predicted Probability')
ax2.set_ylabel('Density')
ax2.set_title('V10 Prediction Distribution')
ax2.legend()

# 3. Confusion Matrix
ax3 = axes[1, 0]
cm = confusion_matrix(y_true, y_pred_binary)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
 xticklabels=['Benign', 'Malignant'],
 yticklabels=['Benign', 'Malignant'])
ax3.set_xlabel('Predicted')
ax3.set_ylabel('Actual')
ax3.set_title(f'Confusion Matrix @ threshold={optimal_threshold:.3f}')

# 4. Version Comparison
ax4 = axes[1, 1]
version_names = list(versions.keys())
x = np.arange(len(version_names))
width = 0.25

aucs = [versions[v]['auc'] for v in version_names]
sens = [versions[v]['sensitivity'] for v in version_names]
spec = [versions[v]['specificity'] for v in version_names]

bars1 = ax4.bar(x - width, aucs, width, label='AUC', color='steelblue')
bars2 = ax4.bar(x, sens, width, label='Sensitivity', color='forestgreen')
bars3 = ax4.bar(x + width, spec, width, label='Specificity', color='orange')

ax4.set_ylabel('Score')
ax4.set_xlabel('Version')
ax4.set_title('Performance Comparison Across Versions')
ax4.set_xticks(x)
ax4.set_xticklabels(version_names)
ax4.legend()
ax4.set_ylim(0, 1.0)
ax4.axhline(0.85, color='red', linestyle='--', alpha=0.5, label='Target AUC')

# Add value labels
for bar in bars1:
 height = bar.get_height()
 ax4.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(config.RESULTS_PATH / 'v10_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n Results saved to {config.RESULTS_PATH / 'v10_results.png'}")

In [ ]:
# SAVE RESULTS

print("="*70)
print(" SAVING RESULTS")
print("="*70)

# Prepare results dictionary
results = {
 'version': 'V10_PAPER_SPEC',
 'timestamp': datetime.now().isoformat(),
 'training_duration': str(training_duration),
 'configuration': {
 'dense_units': config.DENSE_UNITS,
 'effective_batch_size': config.EFFECTIVE_BATCH_SIZE,
 'physical_batch_size': config.BATCH_SIZE,
 'accumulation_steps': config.ACCUMULATION_STEPS,
 'l1_regularization': config.L1_REGULARIZATION,
 'l2_regularization': config.L2_REGULARIZATION,
 'dropout_rate': config.DROPOUT_RATE,
 'use_clahe': config.USE_CLAHE,
 'use_lr_warmup': config.USE_LR_WARMUP,
 'use_tta': config.USE_TTA,
 'tta_augmentations': config.TTA_AUGMENTATIONS,
 'img_size': list(config.IMG_SIZE),
 'gpu_profile': config.GPU_PROFILE
 },
 'test_metrics': {
 'auc_roc': float(auc),
 'optimal_threshold': float(optimal_threshold),
 'sensitivity': float(sensitivity),
 'specificity': float(specificity),
 'accuracy': float(accuracy),
 'ppv': float(ppv),
 'npv': float(npv)
 },
 'confusion_matrix': {
 'tp': int(tp),
 'tn': int(tn),
 'fp': int(fp),
 'fn': int(fn)
 },
 'comparison': versions,
 'improvement_vs_v9': {
 'auc_change': float(auc - versions['V9']['auc']),
 'sensitivity_change': float(sensitivity - versions['V9']['sensitivity']),
 'specificity_change': float(specificity - versions['V9']['specificity'])
 },
 'training_history': histories
}

# Save JSON
results_file = config.RESULTS_PATH / 'v10_results.json'
with open(results_file, 'w') as f:
 json.dump(results, f, indent=2)
print(f"\n Results saved to {results_file}")

# Save CSV summary
csv_data = {
 'Version': ['V10_PAPER_SPEC'],
 'AUC': [auc],
 'Sensitivity': [sensitivity],
 'Specificity': [specificity],
 'Accuracy': [accuracy],
 'PPV': [ppv],
 'NPV': [npv],
 'Threshold': [optimal_threshold],
 'Dense_Units': [config.DENSE_UNITS],
 'Effective_Batch': [config.EFFECTIVE_BATCH_SIZE],
 'L1_L2_Reg': [config.L1_REGULARIZATION],
 'Dropout': [config.DROPOUT_RATE]
}
csv_df = pd.DataFrame(csv_data)
csv_file = config.RESULTS_PATH / 'v10_metrics.csv'
csv_df.to_csv(csv_file, index=False)
print(f" Metrics saved to {csv_file}")

print("\n" + "="*70)

In [ ]:
# FINAL SUMMARY

print("\n" + "="*70)
print(" V10 PAPER-SPEC RESULTS SUMMARY")
print("="*70)

print(f"\n Target vs Achieved:")
print(f" AUC: {auc:.4f} (target: 0.85) {'' if auc >= 0.85 else ''}")
print(f" Sensitivity: {100*sensitivity:.1f}% (target: 85%) {'' if sensitivity >= 0.85 else ''}")
print(f" Specificity: {100*specificity:.1f}% (target: 75%) {'' if specificity >= 0.75 else ''}")

print(f"\n Improvement vs V9:")
print(f" AUC: {auc - versions['V9']['auc']:+.4f}")
print(f" Sensitivity: {(sensitivity - versions['V9']['sensitivity'])*100:+.1f} points")
print(f" Specificity: {(specificity - versions['V9']['specificity'])*100:+.1f} points")

print(f"\n V10 Paper-Spec Implementations:")
print(f" Dense units: {config.DENSE_UNITS} (paper: 2048)")
print(f" Effective batch: {config.EFFECTIVE_BATCH_SIZE} (paper: 32)")
print(f" L1/L2 regularization: {config.L1_REGULARIZATION} (paper: 0.01)")
print(f" Dropout: {config.DROPOUT_RATE} (paper: 0.5)")
print(f" CLAHE preprocessing")
print(f" LR warmup (5 epochs)")
print(f" Gradient accumulation for memory efficiency")

print(f"\n Results saved to:")
print(f" Model: {config.OUTPUT_PATH / 'v10_final_model.h5'}")
print(f" JSON: {config.RESULTS_PATH / 'v10_results.json'}")
print(f" CSV: {config.RESULTS_PATH / 'v10_metrics.csv'}")
print(f" Plot: {config.RESULTS_PATH / 'v10_results.png'}")

print("\n" + "="*70)
print(" V10 PAPER-SPEC TRAINING COMPLETE")
print("="*70)

## V10 Notes

### Key Changes from V9:
1. **Dense units**: 1024 → **2048** (paper specification)
2. **Effective batch**: 8 → **32** (via gradient accumulation)
3. **L1/L2 regularization**: 0.0001 → **0.01** (paper specification)
4. **Dropout**: 0.25 → **0.5** (paper specification)

### Memory Optimization:
- Physical batch = 4 (fits in 12GB VRAM)
- Accumulation steps = 8
- Effective batch = 32 (paper specification achieved!)

### Cache Reuse:
- V10 reuses V9's preprocessed cache
- Same CLAHE preprocessing parameters
- Saves ~10 minutes of preprocessing time

### Expected Results:
- Target: AUC 0.75-0.78 (improved from V9's 0.738)
- Higher regularization may improve generalization
- Larger model capacity (2048 units) may learn better features